# Feature Engineering - Ventas de lácteos 2024

Este notebook construye las variables de entrada (features) necesarias para entrenar el modelo CatBoost de predicción de demanda mensual. El punto de partida es el dataset agregado a nivel mensual generado por `preprocessing.py`.

Dado que disponemos únicamente de 12 meses de histórico por combinación Producto+Presentación, se ha sido conservador en la construcción de features para no consumir demasiadas filas con los lags.

### 1. Importación de bibliotecas y carga de datos

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

In [4]:
INPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed', 'datos_mensuales.csv')
OUTPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed', 'datos_features.csv')

df = pd.read_csv(INPUT_PATH)
df["Fecha_mes"] = pd.to_datetime(df["Fecha_mes"])

print("Shape:", df.shape)
display(df.head())

Shape: (588, 9)


,Categoría,Producto,Presentación,anio,mes,Fecha_mes,cantidad_total_mes,num_pedidos,num_clientes
0,Cremas,Crema agria,Tarrina 200g,2024,1,2024-01-01,12876,29,17
1,Cremas,Crema batida,Spray 250ml,2024,1,2024-01-01,19257,38,17
2,Cremas,Crema de leche,Cartón 946ml,2024,1,2024-01-01,14676,33,17
3,Cremas,Queso crema,Tarrina 250g,2024,1,2024-01-01,16982,35,16
4,Leche,Bebidas lácteas,Botella 1L,2024,1,2024-01-01,3044,6,4


### 2. Lags de la variable objetivo

Los lags capturan el comportamiento reciente de cada serie. Se calculan **dentro de cada grupo** Categoría+Producto+Presentación para evitar contaminación entre series distintas.

- `lag_1` — ventas del mes anterior (el predictor más importante)
- `lag_2` — ventas de hace 2 meses
- `lag_3` — ventas de hace 3 meses

Las primeras 3 filas de cada serie quedarán como `NaN` y se eliminarán, dejando **9 meses** de datos por combinación.

In [5]:
grupo = ["Categoría", "Producto", "Presentación"]

df = df.sort_values(grupo + ["Fecha_mes"]).reset_index(drop=True)

for lag in [1, 2, 3]:
    df[f"lag_{lag}"] = df.groupby(grupo)["cantidad_total_mes"].shift(lag)

display(df[grupo + ["Fecha_mes", "cantidad_total_mes", "lag_1", "lag_2", "lag_3"]].head(15))

,Categoría,Producto,Presentación,Fecha_mes,cantidad_total_mes,lag_1,lag_2,lag_3
0,Cremas,Crema agria,Tarrina 200g,2024-01-01,12876,NaN,NaN,NaN
1,Cremas,Crema agria,Tarrina 200g,2024-02-01,13353,12876.0,NaN,NaN
2,Cremas,Crema agria,Tarrina 200g,2024-03-01,21640,13353.0,12876.0,NaN
3,Cremas,Crema agria,Tarrina 200g,2024-04-01,16396,21640.0,13353.0,12876.0
4,Cremas,Crema agria,Tarrina 200g,2024-05-01,18317,16396.0,21640.0,13353.0
5,Cremas,Crema agria,Tarrina 200g,2024-06-01,12906,18317.0,16396.0,21640.0
6,Cremas,Crema agria,Tarrina 200g,2024-07-01,20483,12906.0,18317.0,16396.0
7,Cremas,Crema agria,Tarrina 200g,2024-08-01,18346,20483.0,12906.0,18317.0
8,Cremas,Crema agria,Tarrina 200g,2024-09-01,15494,18346.0,20483.0,12906.0
9,Cremas,Crema agria,Tarrina 200g,2024-10-01,16539,15494.0,18346.0,20483.0


### 3. Media móvil

La media móvil de los últimos 3 meses suaviza el ruido puntual y proporciona al modelo una visión del nivel reciente de demanda.

In [6]:
df["media_movil_3"] = (
    df.groupby(grupo)["cantidad_total_mes"]
    .transform(lambda x: x.shift(1).rolling(window=3).mean())
)

display(df[grupo + ["Fecha_mes", "cantidad_total_mes", "media_movil_3"]].head(15))

,Categoría,Producto,Presentación,Fecha_mes,cantidad_total_mes,media_movil_3
0,Cremas,Crema agria,Tarrina 200g,2024-01-01,12876,NaN
1,Cremas,Crema agria,Tarrina 200g,2024-02-01,13353,NaN
2,Cremas,Crema agria,Tarrina 200g,2024-03-01,21640,NaN
3,Cremas,Crema agria,Tarrina 200g,2024-04-01,16396,15956.333333
4,Cremas,Crema agria,Tarrina 200g,2024-05-01,18317,17129.666667
5,Cremas,Crema agria,Tarrina 200g,2024-06-01,12906,18784.333333
6,Cremas,Crema agria,Tarrina 200g,2024-07-01,20483,15873.000000
7,Cremas,Crema agria,Tarrina 200g,2024-08-01,18346,17235.333333
8,Cremas,Crema agria,Tarrina 200g,2024-09-01,15494,17245.000000
9,Cremas,Crema agria,Tarrina 200g,2024-10-01,16539,18107.666667


### 4. Variables temporales

El mes del año es un predictor relevante para capturar estacionalidad. Se incluye de dos formas:

- **`mes`** — valor numérico directo (1-12), que CatBoost puede tratar como feature ordinal.
- **`mes_seno` y `mes_coseno`** — codificación cíclica para que el modelo entienda que diciembre (12) y enero (1) son meses consecutivos y no extremos opuestos.

In [7]:
df["mes"] = df["Fecha_mes"].dt.month
df["mes_seno"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_coseno"] = np.cos(2 * np.pi * df["mes"] / 12)

display(df[["Fecha_mes", "mes", "mes_seno", "mes_coseno"]].drop_duplicates().sort_values("mes"))

,Fecha_mes,mes,mes_seno,mes_coseno
0,2024-01-01,1,5.000000e-01,8.660254e-01
1,2024-02-01,2,8.660254e-01,5.000000e-01
2,2024-03-01,3,1.000000e+00,6.123234e-17
3,2024-04-01,4,8.660254e-01,-5.000000e-01
4,2024-05-01,5,5.000000e-01,-8.660254e-01
5,2024-06-01,6,1.224647e-16,-1.000000e+00
6,2024-07-01,7,-5.000000e-01,-8.660254e-01
7,2024-08-01,8,-8.660254e-01,-5.000000e-01
8,2024-09-01,9,-1.000000e+00,-1.836970e-16
9,2024-10-01,10,-8.660254e-01,5.000000e-01


### 5. Eliminación de filas con NaN

Los lags introducen `NaN` en las primeras filas de cada serie. Se eliminan para obtener un dataset limpio listo para el modelo.

In [8]:
filas_antes = len(df)
df = df.dropna().reset_index(drop=True)
filas_despues = len(df)

print(f"Filas eliminadas por NaN: {filas_antes - filas_despues}")
print(f"Filas finales: {filas_despues}")
display(df.head())

Filas eliminadas por NaN: 147
Filas finales: 441


,Categoría,Producto,Presentación,anio,mes,Fecha_mes,cantidad_total_mes,num_pedidos,num_clientes,lag_1,lag_2,lag_3,media_movil_3,mes_seno,mes_coseno
0,Cremas,Crema agria,Tarrina 200g,2024,4,2024-04-01,16396,34,18,21640.0,13353.0,12876.0,15956.333333,8.660254e-01,-0.500000
1,Cremas,Crema agria,Tarrina 200g,2024,5,2024-05-01,18317,36,16,16396.0,21640.0,13353.0,17129.666667,5.000000e-01,-0.866025
2,Cremas,Crema agria,Tarrina 200g,2024,6,2024-06-01,12906,28,16,18317.0,16396.0,21640.0,18784.333333,1.224647e-16,-1.000000
3,Cremas,Crema agria,Tarrina 200g,2024,7,2024-07-01,20483,36,18,12906.0,18317.0,16396.0,15873.000000,-5.000000e-01,-0.866025
4,Cremas,Crema agria,Tarrina 200g,2024,8,2024-08-01,18346,36,16,20483.0,12906.0,18317.0,17235.333333,-8.660254e-01,-0.500000


### 6. Resumen del dataset final

In [9]:
print("Columnas finales:")
print(df.columns.tolist())
print("\nShape:", df.shape)
print("\nMeses por combinación Producto+Presentación:")
display(df.groupby(["Producto", "Presentación"])["Fecha_mes"].count().value_counts())

Columnas finales:
['Categoría', 'Producto', 'Presentación', 'anio', 'mes', 'Fecha_mes', 'cantidad_total_mes', 'num_pedidos', 'num_clientes', 'lag_1', 'lag_2', 'lag_3', 'media_movil_3', 'mes_seno', 'mes_coseno']

Shape: (441, 15)

Meses por combinación Producto+Presentación:


Fecha_mes
9    49
Name: count, dtype: int64

### 7. Guardado

In [11]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset con features guardado en: {OUTPUT_PATH}")

Dataset con features guardado en: c:\Users\esthe\OneDrive\Escritorio\PROJECTS\dairy-demand-forecasting\data\processed\datos_features.csv
